# Bayesian Optimisation: ERK Oscillation (v10 RESUME — warm-start)

Same v10 setup (LED power as BO axis, no light-budget reparam, CNR-screening
FOV finder) but **resumes from a previously-aborted v10 run**.  The
checkpoint cell below loads that run's `bo_results_checkpoint.parquet`
into `agent.df_results`, sets `agent.iteration` so the EI-decay schedule
continues from the right phase, and trims `composed_agent.n_phases` so
only the remaining phases are run.

**Key differences from `bo_erk_oscillation_testv10_led_power.ipynb`:**

1. New `experiment_name` (suffix `_resume`) so the new run writes to a
   separate storage directory and does not overwrite the prior run's
   tracks / OME-Zarr / checkpoints.
2. After the BO agent is constructed, a dedicated cell loads the prior
   run's df_results and warm-starts agent state.  The first phase of
   this resumed session uses GP-acquisition immediately (no
   initial-spread re-scan).
3. **`n_cov_samples=16`** and **`ei_num_samples=4`** (down from 40 / 8 in
   the original v10) to make the robust-acquisition predict cheap
   enough to not be the bottleneck.

**Caveats:**

- Prior FOV / phase indices in `df_results` are shifted to negative
  values + a `source="prior_run"` column is added so any downstream
  group-by-phase plotting can distinguish prior vs resumed rows.
- The FOV finder restarts its well queue from the top of `WELLS`.  With
  `cycle_wells=True` and a fresh `random_seed`, that is fine — wells
  may be revisited but the candidate FOV positions inside them are
  randomised per phase, so resumed phases sample fresh tissue.
- The GP gets re-fit from scratch on the loaded data on the first
  resumed phase — there's no attempt to reload `bo_model.joblib`
  because the kernel hyperparameter posterior is cheap to re-MCMC
  (~4 s) on the existing observations.


In [ ]:
import os
import time
import logging
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Suppress JAX/XLA debug messages
os.environ["JAX_LOG_COMPILES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax

jax.config.update("jax_log_compiles", False)
for _name in list(logging.Logger.manager.loggerDict):
    if "jax" in _name or "absl" in _name:
        logging.getLogger(_name).setLevel(logging.WARNING)

# ---------------------------------------------------------------------------
# gpax/numpyro compatibility shim
# ---------------------------------------------------------------------------
# numpyro >= 0.20 removed haiku support from numpyro.contrib.module, but
# gpax 0.1.9 eagerly imports viDKL via gpax/models/__init__.py, which tries
# `from numpyro.contrib.module import random_haiku_module, haiku_module`.
# We don't use viDKL (only ExactGP, viGP, viSparseGP), so we install stubs
# that raise only if viDKL is actually called. This MUST run before any
# `import gpax` (including the lazy ones inside the BO agent classes).
import numpyro.contrib.module as _ncm


def _haiku_unavailable(*_args, **_kwargs):
    raise NotImplementedError(
        "haiku support was removed from numpyro >= 0.20. "
        "viDKL is not available. Use ExactGP, viGP, or viSparseGP."
    )


if not hasattr(_ncm, "random_haiku_module"):
    _ncm.random_haiku_module = _haiku_unavailable
if not hasattr(_ncm, "haiku_module"):
    _ncm.haiku_module = _haiku_unavailable
# ---------------------------------------------------------------------------

from faro.core.data_structures import (
    PowerChannel,
    SegmentationMethod,
)
from faro.core.controller import Controller
from faro.core.pipeline import ImageProcessingPipeline
from faro.agents.bo_optimization import (
    BO_Parameter,
    BO_Objective,
    BO_Covariate,
)
from faro.agents.bo_oscillation import OscillationBO
import faro.core.utils as utils

## Microscope & pipeline setup

**Microscope:** Jungfrau (no DMD -- full-FOV stimulation).

**Channels:**
- Imaging: miRFP (nuclear marker) + mScarlet3 (ERK-KTR reporter)
- Stimulation: CyanStim (optogenetic activation, power 10)
- Optocheck: mCitrine (verify optoRTK expression, last frame only)

**Pipeline:** CellposeV4 segmentation -> ERK-KTR FE -> Trackpy tracking
-> OptoCheckFE on reference frames. Full-FOV stimulation (no DMD).

In [ ]:
from faro.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")
mic.mmc.setProperty("TIPFSStatus", "State", "On")

In [ ]:
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc  # point the widget at our CMMCore instance
viewer.window.add_dock_widget(mm_wdg)  # dock Micro-Manager controls in napari

In [ ]:
WELLS = []
START_COL = 2
END_COL = 7
for i, row in enumerate("ABCDEFGH"):
    cols = (
        range(START_COL, END_COL + 1)
        if i % 2 == 0
        else range(END_COL, START_COL - 1, -1)
    )
    WELLS.extend(f"{row}{c}" for c in cols)

START_COL = 8
END_COL = 11
for i, row in enumerate("ABCDEFGH"):
    cols = (
        range(START_COL, END_COL + 1)
        if i % 2 == 0
        else range(END_COL, START_COL - 1, -1)
    )
    WELLS.extend(f"{row}{c}" for c in cols)

In [ ]:
WELLS

In [ ]:
len(WELLS)

In [ ]:
# --- Experiment parameters ---
TIME_BETWEEN_TIMESTEPS = 60  # seconds (1 frame/min)

# --- Phase layout (frames; 1 frame == 1 minute at TIME_BETWEEN_TIMESTEPS=60) ---
N_FRAMES_BASELINE = 10
N_FRAMES_STIM = 60
N_FRAMES_RECOVERY = 20

# Derived (don't edit -- adjust the three above instead).
N_FRAMES = N_FRAMES_BASELINE + N_FRAMES_STIM + N_FRAMES_RECOVERY  # 90
FIRST_FRAME_STIM = N_FRAMES_BASELINE  # 10
LAST_FRAME_STIM = FIRST_FRAME_STIM + N_FRAMES_STIM  # 70

# --- Plate calibration ---
PLATE_CALIBRATION_PATH = r"./calib_plate_96.json"  # <-- UPDATE

# --- FOV finder knobs ---
FOV_BORDER_UM = 1000.0
FOV_MIN_DISTANCE_UM = 750.0
FOV_MIN_CELLS = 35
FOV_N_CANDIDATES_PER_WELL = 10

# --- Phased FOV layout ---
N_WELLS_PER_PHASE = 6
FOVS_PER_WELL = 3
N_FOVS = N_WELLS_PER_PHASE * FOVS_PER_WELL  # 18 FOVs per phase
N_CONDITIONS_PER_ITER = 3
FOVS_PER_CONDITION = N_FOVS // N_CONDITIONS_PER_ITER
N_PHASES = 13

WELLS = WELLS[: N_PHASES * N_WELLS_PER_PHASE]


# --- Storage ---
base_path = "E:\\Alex"
experiment_name = "2026-05-07_bo_erk_oscillation_v10_led_power_resume"
path = os.path.join(base_path, experiment_name)

# --- Stimulation channel (base -- exposure is overridden by BO; power fixed) ---
stim_channel = PowerChannel(
    config="CyanStim",
    exposure=100,
    group="TTL_ERK",
    power=10,
)

# --- Imaging channels (acquired every timepoint AND used by FOV finder) ---
imaging_channels = (
    PowerChannel(config="miRFP", exposure=150, group="TTL_ERK", power=95),
    PowerChannel(config="mScarlet3", exposure=150, group="TTL_ERK", power=95),
)

# --- Optocheck channel (acquired on last frame only) ---
optocheck_channel = PowerChannel(
    config="mCitrine",
    exposure=600,
    group="TTL_ERK",
    power=95,
)

In [ ]:
len(WELLS) / N_WELLS_PER_PHASE

In [ ]:
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.segmentation.cellpose_v4 import CellposeV4

# A single Cellpose instance is shared by the experiment pipeline AND the
# FOVFinderAgent below.  Reusing the same model avoids loading it twice
# (memory + GPU savings) and guarantees that the FOV finder counts cells
# with exactly the same segmentation the experiment will use.
segmentator = CellposeV4(
    custom_model_path="E:\\models\\cellpose\\LifeActH2B_mixed_with_only_H2B_v1",
    min_size=100,
)

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=[
        SegmentationMethod(
            name="labels",
            segmentation_class=segmentator,
            use_channel=0,  # segment on miRFP (nuclear marker)
            save_tracked=True,
        )
    ],
    feature_extractor=FE_ErkKtr("labels"),
    tracker=TrackerTrackpy(search_range=50),
    stimulator=StimWholeFOV(),
    feature_extractor_ref=OptoCheckFE(used_mask="labels"),
)

from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(storage_path=path)

## FOV selection (per-phase, automated)

Instead of manually selecting 18 positions in napari, a `FOVFinderAgent`
(plugged into the generic `ComposedAgent`) picks fresh positions before
**every phase**:

1. The agent loads the plate calibration (`WellPlatePlan` JSON saved by
   the pymmcore-widgets MDA plate widget).
2. It pops the next `N_WELLS_PER_PHASE` (= 6) wells from `WELLS`.
3. For each well it generates `FOV_N_CANDIDATES_PER_WELL` (= 8) random
   candidate positions, kept `FOV_BORDER_UM` µm away from the well edge
   and at least `FOV_MIN_DISTANCE_UM` apart from each other.
4. It snaps the candidates with the segmentation channel only (miRFP)
   via `mic.run_mda` and segments the result with the **same**
   `CellposeV4` instance the experiment pipeline uses (`segmentator`).
5. Candidates with `< FOV_MIN_CELLS` cells are dropped; the remaining
   ones are reduced to `FOVS_PER_WELL` (= 3) per well by greedy
   farthest-point sampling so the picked FOVs do not overlap.

The result (18 fresh FOVs / phase) is fed straight into
`OscillationBO.run_one_phase`, which is wired up by the `ComposedAgent`
in the next sections.

The segmentator was already created in the pipeline cell above and is
reused here -- no extra setup needed.  The `FOVFinderAgent` is constructed
in the *Configure and run* section together with the BO agent and the
composed driver.


## Oscillation classifier

Load the pre-trained sliding-window oscillation classifier. This model
uses FFT, ACF, and time-domain features extracted from the ERK-KTR
`cnr` trace to classify each window as oscillating or not.

The classifier is restricted at runtime to the **stimulation window
only** (via `OscillationBO.classifier_window`, which defaults to
`(FIRST_FRAME_STIM, LAST_FRAME_STIM)`). Baseline and recovery frames
are acquired but not classified.

The BO target in this notebook is **`frac_responders`** — per FOV, the
fraction of valid cells whose **mean** classifier osc-probability across
the scoring window is `>= frac_responder_threshold` (default 0.75).
This is computed *directly* from the classifier's per-window
probabilities; the legacy 3-gate `frac_oscillating` rule (FFT amplitude
+ max probability + consecutive windows) is **not** used by the BO and
is disabled in the cell below by setting all three thresholds to 0.

Cells are only included in the responder calculation if they pass two
quality gates:
- Tracked for `>= min_track_fraction * n_frames` (default 80 %)
- Baseline CNR `< max_baseline_cnr` (default 1.0)


In [ ]:
import joblib

# --- Load pre-trained oscillation classifier ---
CLASSIFIER_PATH = r"./oscillation_model_60min.joblib"  # <-- UPDATE THIS PATH

model_data = joblib.load(CLASSIFIER_PATH)
osc_clf = model_data["clf"]
osc_scaler = model_data["scaler"]
osc_feature_cols = model_data["feature_cols"]
osc_cfg = model_data["config"]
osc_cfg["window_size"] = model_data["window_size"]
osc_cfg["window_step"] = model_data["window_step"]

# Import the predict_trace function from the classifier script
_classifier_script = "./apply_oscillation_classifier_v2.py"

_spec = importlib.util.spec_from_file_location("osc_classifier", _classifier_script)
_osc_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_osc_module)
predict_trace = _osc_module.predict_trace

# --- Oscillation thresholds ---
# A cell is oscillating iff ALL three gates pass (>= for all):
#   1. FFT amplitude score >= 0.3
#   2. Max osc. probability >= 0.95
#   3. Consecutive osc. windows >= 3
MIN_OSC_PROBABILITY = (
    0.0  # v6: gates disabled, BO target is frac_responders (mean prob >= 0.75 per cell)
)
MIN_CONSECUTIVE_WINDOWS = 0
MIN_FFT_AMPLITUDE = 0.0

print(f"Loaded oscillation classifier from {CLASSIFIER_PATH}")
print(f"  Window: {osc_cfg['window_size']} steps, stride {osc_cfg['window_step']}")
print(
    f"  Thresholds: prob >= {MIN_OSC_PROBABILITY}, "
    f"consecutive >= {MIN_CONSECUTIVE_WINDOWS}, "
    f"fft_amplitude >= {MIN_FFT_AMPLITUDE}"
)

## Batch BO agent

The :class:`OscillationBO` subclass of :class:`BOptGPAX` lives in
``faro.agents.bo_oscillation``.  It overrides:

- ``_create_events_for_batch`` -- builds RTMEvents for ``N_FOVS`` FOVs across
  ``N_CONDITIONS_PER_ITER`` conditions (``FOVS_PER_CONDITION`` each), applying
  per-frame ramped stim exposure.
- ``_preprocess_results`` -- runs the oscillation classifier on each cell's
  ``cnr`` trace (sliced to the stimulation window via ``classifier_window``)
  and computes the per-FOV oscillating fraction. Also computes the
  ``baseline_cnr`` covariate from the first ``n_baseline_frames``.
- ``_on_phase_complete`` -- live plotting + checkpoint saving + saves the
  trained GP to ``{storage_path}/bo_model.joblib`` for post-hoc analysis.

All batch-BO machinery (sequential greedy acquisition with local penalisation,
FOV-index offsetting across phases, ``run_one_phase`` integration with
``ComposedAgent``) is inherited from ``BOptGPAX``.  See
[faro/agents/bo_oscillation.py](../../faro/agents/bo_oscillation.py).


## Configure and run Bayesian Optimisation

**Parameter space (v10 — direct (exposure, ramp, pulse_interval, power)):**

- `stim_exposure` (ms): 25–1000 step 25 = 40 levels (per-pulse base
  exposure).  Per-pulse delivery is also clipped at 1000 ms in
  `_create_events_for_batch` so ramp can never push a single pulse
  above the hardware ceiling.
- `ramp` (ms/pulse): 0–50 step 10 = 6 levels.
- `pulse_interval` (frames): 1–10 step 2 = 5 levels ⇒ {1, 3, 5, 7, 9}.
- `led_power` (%): 10–100 step 10, **`log_scale=True`** = 10 levels.
  The GP scaler takes `log(led_power)` so the kernel sees a near-linear
  response axis (biology saturates above ~30–40 %).

Total grid: 40 × 6 × 5 × 10 = **12 000 candidate conditions**.

**Initial exploration cap (`initial_exploration_cap=0.7`):**
First 2 phases restrict to `stim_exposure ≤ ~700 ms`, `ramp ≤ 35`,
`pulse_interval ≤ 7`, `led_power ≤ ~73 %` (cap applied per axis on
fraction-of-range; integer params snap to the next grid step).

**Covariates:** `n_cells`, `optortk_expression` (log-scaled),
`baseline_cnr`. Marginalised over during acquisition.

**Default objective: `frac_responders`** — per-FOV fraction of cells
whose per-cell mean classifier probability across the scoring window is
`≥ 0.75`.  Same as v9.

**FOV finder pre-screen.**  Same as v11: candidate FOVs are imaged with
miRFP + mScarlet3, segmented, run through `FE_ErkKtr`, and rejected
unless ≥ 75 % of cells have baseline CNR < 1.0.


In [ ]:
import dataclasses

from faro.agents import FOVFinderAgent, ComposedAgent, FOVCondition
from faro.agents.bo_oscillation import MIN_STIM_EXPOSURE_MS
from faro.core.data_structures import RTMSequence
import faro.core.utils as faro_utils

# ---------------------------------------------------------------------------
# v10: led_power as a 4th BO control axis, no light-budget reparam
# ---------------------------------------------------------------------------
# OscillationBO speaks (stim_exposure, ramp, pulse_interval) and uses
# self.stim_channel for ALL conditions.  v10 adds led_power as a BO axis
# and constructs a fresh PowerChannel per condition with the BO-chosen
# power, so the microscope receives the right intensity for each
# acquisition.  Per-pulse exposure is also clipped at MAX_STIM_EXPOSURE_MS
# (1000 ms) so ramp can never push a single pulse above the user-
# configured ceiling.


class PoweredOscillationBO(OscillationBO):
    """OscillationBO subclass that adds led_power as a BO control axis."""

    MAX_STIM_EXPOSURE_MS: float = 1000.0

    def _create_events_for_batch(self, param_list):
        # Replicates OscillationBO._create_events_for_batch with two diffs:
        #   (1) per-pulse exposure CLIPPED at MAX_STIM_EXPOSURE_MS in
        #       addition to the >=25 ms floor inherited from MIN_STIM_EXPOSURE_MS.
        #   (2) a fresh PowerChannel with the BO-chosen led_power is
        #       constructed per condition instead of reusing self.stim_channel.
        phase_id = self._phase_counter
        all_events = []
        fovs_per_condition = self.fovs_per_condition
        self._current_condition_map = {}

        for cond_idx, params in enumerate(param_list):
            start = cond_idx * fovs_per_condition
            end = start + fovs_per_condition
            cond_positions = self.fov_positions[start:end]

            for fov_idx in range(start, end):
                self._current_condition_map[fov_idx + self._fov_index_offset] = params

            base_exp = float(params["stim_exposure"])
            ramp = float(params["ramp"])
            pulse_interval = max(1, int(params.get("pulse_interval", 1)))
            led_power = int(round(float(params["led_power"])))

            all_frames = list(
                range(self.first_frame_stim, self.last_frame_stim, pulse_interval)
            )
            # Cap per-pulse exposure at MAX (1000 ms); pulses driven above
            # the cap by ramp would otherwise risk hardware unreliability.
            all_exposures = [
                min(base_exp + ramp * i, self.MAX_STIM_EXPOSURE_MS)
                for i in range(len(all_frames))
            ]

            kept = [
                (f, e)
                for f, e in zip(all_frames, all_exposures)
                if e >= MIN_STIM_EXPOSURE_MS
            ]
            n_dropped = len(all_frames) - len(kept)
            if n_dropped > 0:
                print(
                    f"  Cond {cond_idx}: dropped {n_dropped}/{len(all_frames)} "
                    f"sub-{MIN_STIM_EXPOSURE_MS:g}ms pulses"
                )

            if not kept:
                print(
                    f"  Cond {cond_idx}: ALL {len(all_frames)} pulses below "
                    f"{MIN_STIM_EXPOSURE_MS:g}ms floor -- emitting NO stim "
                    f"events for this condition (imaging only)"
                )
                stim_frames = frozenset()
                exposures = ()
            else:
                stim_frames = frozenset(f for f, _ in kept)
                exposures = tuple(e for _, e in kept)

            # Per-condition stim channel with the BO-chosen LED power.
            cond_stim_channel = dataclasses.replace(self.stim_channel, power=led_power)

            acq = RTMSequence(
                time_plan={
                    "interval": self.time_between_timesteps,
                    "loops": self.n_frames,
                },
                stage_positions=cond_positions,
                channels=self.imaging_channels,
                stim_channels=(cond_stim_channel,),
                stim_frames=stim_frames,
                stim_exposure=exposures,
                ref_channels=(self.optocheck_channel,),
                ref_frames=frozenset({self.n_frames - 1}),
                rtm_metadata={
                    "phase_name": f"BO_iter_{phase_id}_cond_{cond_idx}",
                    "phase_id": phase_id,
                    "condition_idx": cond_idx,
                    "stim_exposure": base_exp,
                    "ramp": ramp,
                    "pulse_interval": pulse_interval,
                    "led_power": led_power,
                },
            )

            p_offset = cond_idx * fovs_per_condition + self._fov_index_offset
            for ev in acq:
                local_p = ev.index.get("p", 0)
                all_events.append(
                    ev.model_copy(
                        update={"index": {**dict(ev.index), "p": local_p + p_offset}}
                    )
                )

        all_events = faro_utils.apply_fov_batching(all_events, time_per_fov=2.0)

        print(f"  Created {len(all_events)} events for phase {phase_id}")
        for i, p in enumerate(param_list):
            exp_start = p["stim_exposure"]
            pi = max(1, int(p.get("pulse_interval", 1)))
            n_pulses_i = len(range(self.first_frame_stim, self.last_frame_stim, pi))
            exp_end = min(
                exp_start + p["ramp"] * (n_pulses_i - 1), self.MAX_STIM_EXPOSURE_MS
            )
            fov_start = i * fovs_per_condition + self._fov_index_offset
            fov_end = fov_start + fovs_per_condition - 1
            print(
                f"    Cond {i} (FOVs {fov_start}-{fov_end}): "
                f"stim_exposure={exp_start:.0f}ms, ramp={p['ramp']:.0f}ms/pulse, "
                f"pulse_interval={pi} frames, led_power={int(round(p['led_power']))}%, "
                f"n_pulses={n_pulses_i} (final exposure clipped at 1000ms: {exp_end:.0f}ms)"
            )
        return all_events

    def _preprocess_results(self, fov_tracks):
        df = super()._preprocess_results(fov_tracks)
        if df.empty:
            return df
        # Inject led_power per FOV from _current_condition_map so the BO
        # machinery (which reads df[param.name]) can recover it.
        powers = []
        for fov_idx in df["fov"].astype(int):
            params = self._current_condition_map.get(int(fov_idx), {})
            powers.append(int(round(float(params.get("led_power", 0)))))
        df["led_power"] = powers
        return df


# --- BO parameters ----------------------------------------------------------
# Order matters: x_total_linespace is built via np.meshgrid(..., indexing="ij")
# so the FIRST parameter varies SLOWEST in the linearised grid.  The plot
# code below uses pulse_interval (3rd axis) as the panel grid and pins
# led_power (4th axis) at its global-best value when rendering surfaces.
bo_params = [
    BO_Parameter(name="stim_exposure", bounds=(25.0, 1000.0), spacing=25.0),
    BO_Parameter(name="ramp", bounds=(0.0, 50.0), spacing=10.0),
    BO_Parameter(
        name="pulse_interval",
        bounds=(1.0, 10.0),
        spacing=2.0,
        param_type="int",
    ),
    BO_Parameter(
        name="led_power",
        bounds=(10.0, 100.0),
        spacing=10.0,
        param_type="int",
        log_scale=True,  # GP scaler sees log(power) -- matches LED saturation curve.
    ),
]

# --- Covariates (observed, not controlled) ---
bo_covariates = [
    BO_Covariate(name="n_cells"),
    BO_Covariate(name="optortk_expression", log_scale=True),
    BO_Covariate(name="baseline_cnr"),
]

# --- Objective ---
bo_objective = BO_Objective(name="frac_responders", goal="maximize")

# --- Pre-phase agent: FOV finder with CNR pre-screen --------------------
# Identical to v11: image miRFP + mScarlet3, run FE_ErkKtr per candidate,
# reject FOVs where < 75 % of cells have baseline CNR < 1.0.
fov_finder = FOVFinderAgent(
    microscope=mic,
    well_plate_plan=PLATE_CALIBRATION_PATH,
    wells=WELLS,
    wells_per_phase=N_WELLS_PER_PHASE,
    fovs_per_well=FOVS_PER_WELL,
    n_candidates_per_well=FOV_N_CANDIDATES_PER_WELL,
    border_um=FOV_BORDER_UM,
    min_distance_um=FOV_MIN_DISTANCE_UM,
    min_cells=FOV_MIN_CELLS,
    max_cells=250,
    imaging_channels=imaging_channels,
    segmentator=segmentator,
    seg_channel_index=0,
    feature_extractor=FE_ErkKtr("labels"),
    fov_conditions=[
        FOVCondition("cnr", "below", 1.0, min_fraction=0.75),
    ],
    random_seed=None,
    selection_mode="extremes",
    cycle_wells=True,
    verbose=False,
    z=None,
)
print(
    f"FOVFinder: {len(WELLS)} wells queued -> "
    f"{fov_finder.n_remaining_phases} phases @ {N_WELLS_PER_PHASE} wells/phase"
    f"  (cycle_wells=True: refills indefinitely)"
)
print("  Pre-screen: FE_ErkKtr + FOVCondition(cnr<1.0, min_fraction=0.75)")

# --- BO agent ---
# About `use_closed_form_predict=True`:
# Picks the closed-form posterior mean + diag(var) computation in
# `_compute_robust_acq` instead of gpax's MC-sampled predict_in_batches.
# ~10x faster per acquisition call on a 14k-row grid x 16 covariate
# samples.  Requires the gp kernel to be stationary (Matern / RBF /
# Periodic — all good here) and `mean_fn=None` (default).  If you ever
# swap in a non-stationary kernel or add a prior mean function, flip
# this to False to fall back to the legacy MC predict path.  See
# BOptGPAX._predict_mean_diag_var_closed_form for the full list of
# assumptions and limitations.
agent = PoweredOscillationBO(
    storage_path=path,
    n_frames=N_FRAMES,
    first_frame_stim=FIRST_FRAME_STIM,
    last_frame_stim=LAST_FRAME_STIM,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    imaging_channels=imaging_channels,
    stim_channel=stim_channel,
    optocheck_channel=optocheck_channel,
    osc_clf=osc_clf,
    osc_scaler=osc_scaler,
    osc_feature_cols=osc_feature_cols,
    osc_cfg=osc_cfg,
    osc_predict_fn=predict_trace,
    min_osc_probability=MIN_OSC_PROBABILITY,
    min_consecutive_windows=MIN_CONSECUTIVE_WINDOWS,
    min_fft_amplitude=MIN_FFT_AMPLITUDE,
    n_baseline_frames=N_FRAMES_BASELINE,
    parameters_to_optimize=bo_params,
    objective_metric=bo_objective,
    bo_covariates=bo_covariates,
    n_iterations=N_PHASES,
    n_conditions_per_iter=N_CONDITIONS_PER_ITER,
    n_initial_phases=2,
    initial_exploration_cap=0.7,
    frac_responder_threshold=0.75,
    acquisition_function="ei",
    n_cov_samples=16,  # was 40 in v9 — dropped to 16 to halve robust-acq predict cost.
    ei_xi=0.1,
    ei_xi_final=0.01,
    ei_num_samples=4,  # was 8 in v9 — dropped to 4 (still 4*num_mcmc=3200 effective draws).
    ei_xi_decay_fraction=0.7,
    use_closed_form_predict=True,  # ~10x faster acquisition; see comment above + class docstring.
    verbose=True,
    max_baseline_cnr=1.00,
    min_track_fraction=0.8,
)

composed_agent = ComposedAgent(
    inner_agent=agent,
    pre_phase_agents=[fov_finder],
    n_phases=N_PHASES,
)

ctrl = Controller(mic, pipeline, writer=writer, agent=composed_agent)

print(f"Parameter grid: {len(agent.x_total_linespace)} candidate conditions")
print(
    f"Phases: {N_PHASES}  ({agent.n_initial_phases} initial-spread + "
    f"{N_PHASES - agent.n_initial_phases} BO batches)"
)
print(
    f"Conditions per phase: {N_CONDITIONS_PER_ITER}  "
    f"FOVs per condition: {FOVS_PER_CONDITION}  Total FOVs/phase: {N_FOVS}"
)
print(f"Total FOV observations after {N_PHASES} phases: ~{N_PHASES * N_FOVS}")
print(f"BO objective: {bo_objective.name} (goal={bo_objective.goal})")

In [ ]:
# ---------------------------------------------------------------------------
# RESUME from a previous (aborted) v10 run
# ---------------------------------------------------------------------------
# Pre-loads the prior run's cumulative checkpoint into agent.df_results and
# advances agent.iteration so initial-spread phases are skipped.  The
# composed_agent's phase count is trimmed to only the remaining phases.

import pandas as pd

# --- USER-ADJUSTABLE ---------------------------------------------------
# Path to the storage_path of the aborted run (i.e. the directory that
# contains bo_results_checkpoint.parquet).  Update both lines if needed.
PRIOR_RUN_PATH = r"E:\\Alex\\2026-05-07_bo_erk_oscillation_v10_led_power"
# -----------------------------------------------------------------------

prior_ckpt = os.path.join(PRIOR_RUN_PATH, "bo_results_checkpoint.parquet")
if not os.path.exists(prior_ckpt):
    raise FileNotFoundError(
        f"Prior run checkpoint not found:\n  {prior_ckpt}\n"
        "Adjust PRIOR_RUN_PATH above to point at the aborted run's "
        "storage directory."
    )

prior_df = pd.read_parquet(prior_ckpt)
if prior_df.empty:
    raise RuntimeError(
        f"Prior run checkpoint at {prior_ckpt} is empty -- nothing to resume from."
    )

n_prior_phases = int(prior_df["phase_id"].max()) + 1
print(
    f"Loaded prior run: {len(prior_df)} FOV observations across "
    f"{n_prior_phases} phases\n  from: {prior_ckpt}"
)

# Shift prior phase_ids to negative values so they do not collide with the
# resumed session's phase_ids (which composed_agent will count from 0).
# Also tag the source so downstream plots can filter prior vs resumed.
prior_df = prior_df.copy()
prior_df["phase_id"] = prior_df["phase_id"] - n_prior_phases
prior_df["source"] = "prior_run"

# Inject into the agent BEFORE any phases run.
agent.df_results = prior_df
agent.x = agent._extract_x_from_df(prior_df)
agent.y = agent._extract_y_from_df(prior_df)

# agent.iteration is used both by the initial-spread gate AND by the
# EI-decay schedule (current_ei_xi).  Setting it to n_prior_phases makes
# the first resumed phase pick up the EI-decay schedule at exactly the
# right point.
agent.iteration = n_prior_phases

# Reconstruct the "performed experiments" array (used by the local-
# penalisation step in batch BO) so the very first resumed phase still
# applies the inverse-distance penalty against prior picks.
#
# IMPORTANT: only the FIRST n_ctrl columns are kept.  The runtime
# appends new picks as control-axis-only vectors (see
# `next_parameters.reshape(1, -1)` in BOptGPAX._determine_next_parameters)
# and `_compute_robust_acq` slices `[:, :n_ctrl]` for the penalty.
# Including covariate columns here would cause the very first
# np.concatenate to fail with a "size 7 vs 4" shape mismatch on the
# first batch pick.
n_ctrl = len(agent.parameters_to_optimize)
agent.x_performed_experiments = agent._extract_x_from_df(prior_df)[:, :n_ctrl].copy()

# Remove any prior-measured points from x_unmeasured so the BO does not
# repeat them.  Match by control-axis values only (covariates are
# observed, not picked).
prior_ctrl_pts = prior_df[[p.name for p in agent.parameters_to_optimize]].to_numpy(
    dtype=float
)
keep_mask = np.ones(len(agent.x_unmeasured), dtype=bool)
for pt in prior_ctrl_pts:
    diffs = np.abs(agent.x_unmeasured - pt).sum(axis=1)
    j = int(np.argmin(diffs))
    if diffs[j] < 1e-6:
        keep_mask[j] = False
agent.x_unmeasured = agent.x_unmeasured[keep_mask]
print(
    f"  x_unmeasured trimmed: {(~keep_mask).sum()} prior conditions removed; "
    f"{len(agent.x_unmeasured)} candidates remain on the BO grid"
)

# Trim composed_agent.n_phases to remaining target.
remaining = N_PHASES - n_prior_phases
if remaining <= 0:
    raise RuntimeError(
        f"Prior run already has {n_prior_phases} >= N_PHASES ({N_PHASES}) "
        "completed phases -- nothing to resume.  Either bump N_PHASES "
        "above or rerun this notebook with a different prior path."
    )
composed_agent.n_phases = remaining
print(
    f"Resuming with {remaining} more phases "
    f"({n_prior_phases} completed + {remaining} new = {N_PHASES} total)."
)
print(
    f"  agent.iteration = {agent.iteration}  "
    f"(EI decay schedule continues from this point)"
)
print(f"  bo objective = {agent.objective_metric.name}")

In [ ]:
# 4D-aware live plot override (stim_exposure x ramp x pulse_interval x led_power).
#
# For each phase we render a 3-row x N(pulse_interval) panel grid:
#   row 0 -- measured frac_responders (per-FOV scatter; led_power encoded
#            as marker edge intensity if matplotlib supports it, else
#            shown as colour bar on a side legend).
#   row 1 -- GP-predicted landscape (stim_exposure x ramp at fixed
#            pulse_interval AND led_power = GP-global-best level).
#   row 2 -- acquisition surface, same slicing.
#
# led_power is intentionally NOT panelled (would give a 5x10 grid).
# Instead the title of each panel announces which led_power slice is
# rendered, and a separate 1-D summary (added to the post-run viz cells)
# shows the marginalised effect of led_power.
import types
import matplotlib.pyplot as _plt


_AXIS_LABELS = {
    "stim_exposure": "stim_exposure (ms)",
    "ramp": "ramp (ms/pulse)",
    "pulse_interval": "pulse_interval (frames)",
    "led_power": "led_power (%)",
}


def _axis_label(name):
    return _AXIS_LABELS.get(name, name)


def _slice_grid_at(self, ctrl_grid_4d, fix_idx, fix_value):
    """Return a 2D (n_p1, n_p2) view of ctrl_grid_4d at fixed third+fourth axes.

    Used to slice both the GP-predicted landscape and the acquisition surface.
    """
    pass  # unused -- kept here for documentation; logic inlined below.


def _plot_landscape_and_acq_4d(
    self,
    ctx,
    df_results,
    ax_mean,
    ax_acq,
    fig,
    *,
    pulse_interval_level,
    led_power_level,
    p1_name,
    p2_name,
):
    gp_model = ctx["gp_model"]
    x_scaler = ctx["x_scaler"]
    y_scaler = ctx["y_scaler"]
    rng_key_predict = ctx["rng_key_predict"]
    acq_values_total = ctx["acq_values_total"]
    acquisition_used = ctx["acquisition_used"]
    x_unmeasured = ctx["x_unmeasured_at_computation"]

    x_total_ctrl = self.x_total_linespace.copy()
    unique_x1 = np.unique(x_total_ctrl[:, 0])
    unique_x2 = np.unique(x_total_ctrl[:, 1])
    n_ctrl = len(unique_x1) * len(unique_x2)

    # Build the 4D ctrl_grid with axes 3 (pulse_interval) and 4 (led_power)
    # pinned at the requested levels.  Order matches np.meshgrid(..., "ij")
    # + flatten on the BO parameter grid.
    ctrl_grid = np.array(
        [
            [x1, x2, pulse_interval_level, led_power_level]
            for x1 in unique_x1
            for x2 in unique_x2
        ]
    )

    if len(self.bo_covariates) > 0 and not df_results.empty:
        cov_cols = [c.name for c in self.bo_covariates]
        cov_vals_full = np.asarray(df_results[cov_cols].to_numpy(), dtype=float)
        n_cov_samples = 50
        _plot_rng = np.random.default_rng(0)
        row_idx = _plot_rng.integers(0, cov_vals_full.shape[0], size=n_cov_samples)
        cov_samples_joint = cov_vals_full[row_idx]
    else:
        n_cov_samples = 1
        cov_samples_joint = None

    if cov_samples_joint is not None:
        x_grid_full = np.hstack(
            [
                np.repeat(ctrl_grid, n_cov_samples, axis=0),
                np.tile(cov_samples_joint, (n_ctrl, 1)),
            ]
        )
    else:
        x_grid_full = ctrl_grid

    x_grid_scaled = x_scaler.transform(x_grid_full)

    from faro.agents.bo_optimization_sparse import _safe_batch_size

    n_rows = np.asarray(x_grid_scaled).shape[0]
    _bs = _safe_batch_size(n_rows, 1000)
    y_pred_scaled, _ = gp_model.predict_in_batches(
        rng_key_predict,
        x_grid_scaled,
        batch_size=_bs,
        n=self.ei_num_samples,
        noiseless=True,
    )
    y_pred = y_scaler.inverse_transform(
        np.asarray(y_pred_scaled).reshape(-1, 1)
    ).flatten()

    if len(self.bo_covariates) > 0:
        y_pred_marg = y_pred.reshape(n_ctrl, n_cov_samples).mean(axis=1)
    else:
        y_pred_marg = y_pred

    X_mesh, Y_mesh = np.meshgrid(unique_x1, unique_x2, indexing="ij")
    y_pred_2d = y_pred_marg.reshape(len(unique_x1), len(unique_x2))

    level_label = (
        f"\npulse_interval={int(pulse_interval_level)}, "
        f"led_power={int(led_power_level)}%"
    )
    overlay_df = df_results[
        (df_results["pulse_interval"].round().astype(int) == int(pulse_interval_level))
        & (df_results["led_power"].round().astype(int) == int(led_power_level))
    ]

    im1 = ax_mean.pcolormesh(X_mesh, Y_mesh, y_pred_2d, cmap="viridis", shading="auto")
    fig.colorbar(im1, ax=ax_mean, label=f"predicted {self.objective_metric.name}")
    if not overlay_df.empty:
        ax_mean.scatter(
            overlay_df[p1_name],
            overlay_df[p2_name],
            c="white",
            s=15,
            alpha=0.6,
            marker="x",
            linewidths=0.8,
        )
    ax_mean.set_xlabel(_axis_label(p1_name))
    ax_mean.set_ylabel(_axis_label(p2_name))
    ax_mean.set_title(f"GP predicted landscape{level_label}")

    # Acquisition: pad to full ctrl grid then mask on the two pinned axes.
    acq_marg = np.asarray(acq_values_total)
    full_n = len(x_total_ctrl)
    if len(acq_marg) != full_n:
        acq_full = np.zeros(full_n)
        for j, pt in enumerate(x_unmeasured):
            diffs = np.abs(x_total_ctrl - pt).sum(axis=1)
            idx = np.argmin(diffs)
            if j < len(acq_marg):
                acq_full[idx] = float(acq_marg[j])
        acq_marg = acq_full

    mask = (np.round(x_total_ctrl[:, 2]).astype(int) == int(pulse_interval_level)) & (
        np.round(x_total_ctrl[:, 3]).astype(int) == int(led_power_level)
    )
    acq_2d = acq_marg[mask].reshape(len(unique_x1), len(unique_x2))

    im2 = ax_acq.pcolormesh(X_mesh, Y_mesh, acq_2d, cmap="inferno", shading="auto")
    acq_label = acquisition_used.upper()
    fig.colorbar(im2, ax=ax_acq, label=f"{acq_label} acquisition")
    if not overlay_df.empty:
        ax_acq.scatter(
            overlay_df[p1_name],
            overlay_df[p2_name],
            c="white",
            s=15,
            alpha=0.6,
            marker="x",
            linewidths=0.8,
        )

    picks = getattr(self, "_current_batch_picks", None)
    if picks:
        picks_arr = np.array(picks, dtype=float)
        if picks_arr.shape[1] >= 4:
            level_mask_arr = (
                np.round(picks_arr[:, 2]).astype(int) == int(pulse_interval_level)
            ) & (np.round(picks_arr[:, 3]).astype(int) == int(led_power_level))
            picks_arr = picks_arr[level_mask_arr]
        if len(picks_arr) > 0:
            ax_acq.scatter(
                picks_arr[:, 0],
                picks_arr[:, 1],
                c="cyan",
                s=120,
                marker="X",
                edgecolors="k",
                linewidths=1.0,
                zorder=10,
                label="next conditions",
            )
            ax_acq.legend(loc="upper right", fontsize=7)

    ax_acq.set_xlabel(_axis_label(p1_name))
    ax_acq.set_ylabel(_axis_label(p2_name))
    ax_acq.set_title(f"Acquisition {acq_label}{level_label}")


def _pick_led_power_for_plot(self, df_results):
    """Pin led_power for surface rendering at the global-best observed value.

    Falls back to the grid median while we have no observations yet.
    """
    if (
        df_results is not None
        and not df_results.empty
        and "led_power" in df_results.columns
    ):
        obj = self.objective_metric.name
        # mean over FOVs at each led_power, take the argmax level.
        best = df_results.groupby("led_power")[obj].mean().idxmax()
        return int(round(float(best)))
    levels = sorted(np.unique(self.x_total_linespace[:, 3]).tolist())
    return int(round(float(levels[len(levels) // 2])))


def _plot_live_4d(self, df_results, iteration_label, save_subdir="after"):
    p1_name = self.parameters_to_optimize[0].name  # stim_exposure
    p2_name = self.parameters_to_optimize[1].name  # ramp
    pi_levels = sorted(np.unique(self.x_total_linespace[:, 2]).astype(int).tolist())
    n_cols = len(pi_levels)

    led_power_pin = _pick_led_power_for_plot(self, df_results)

    fig, axes = _plt.subplots(
        3, n_cols, figsize=(6 * n_cols, 14), dpi=200, squeeze=False
    )
    fig.suptitle(
        f"{iteration_label}\n"
        f"GP/acq surfaces pinned at led_power = {led_power_pin}%",
        fontsize=12,
        fontweight="bold",
    )

    obj_name = self.objective_metric.name
    ctx = getattr(self, "_last_plot_context", None)

    p1_step = float(self.parameters_to_optimize[0].spacing or 1.0)
    p2_step = float(self.parameters_to_optimize[1].spacing or 1.0)

    for col, pi in enumerate(pi_levels):
        ax_meas = axes[0, col]
        ax_landscape = axes[1, col]
        ax_acq = axes[2, col]

        df_slice = df_results[
            df_results["pulse_interval"].round().astype(int) == int(pi)
        ]

        if not df_slice.empty:
            jitter_x = self._rng.normal(0, p1_step * 0.15, size=len(df_slice))
            jitter_y = self._rng.normal(0, p2_step * 0.15, size=len(df_slice))
            sc = ax_meas.scatter(
                df_slice[p1_name].values + jitter_x,
                df_slice[p2_name].values + jitter_y,
                c=df_slice[obj_name],
                cmap="viridis",
                s=30,
                edgecolors="k",
                linewidths=0.3,
                alpha=0.8,
            )
            fig.colorbar(sc, ax=ax_meas, label=obj_name)
        ax_meas.set_xlabel(_axis_label(p1_name))
        ax_meas.set_ylabel(_axis_label(p2_name))
        ax_meas.set_title(f"Measured {obj_name}\npulse_interval={pi} (all led_power)")

        if ctx is None:
            for ax in (ax_landscape, ax_acq):
                ax.text(
                    0.5,
                    0.5,
                    "GP not fit yet\n(initial batch)",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    fontsize=11,
                    color="gray",
                )
                ax.set_xlabel(_axis_label(p1_name))
                ax.set_ylabel(_axis_label(p2_name))
            ax_landscape.set_title(
                f"GP predicted landscape\npulse_interval={pi}, led_power={led_power_pin}%"
            )
            ax_acq.set_title(
                f"Acquisition\npulse_interval={pi}, led_power={led_power_pin}%"
            )
        else:
            try:
                _plot_landscape_and_acq_4d(
                    self,
                    ctx,
                    df_results,
                    ax_landscape,
                    ax_acq,
                    fig,
                    pulse_interval_level=pi,
                    led_power_level=led_power_pin,
                    p1_name=p1_name,
                    p2_name=p2_name,
                )
            except Exception as e:
                for ax in (ax_landscape, ax_acq):
                    ax.text(
                        0.5,
                        0.5,
                        f"Plot failed:\n{type(e).__name__}: {e}",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                        fontsize=9,
                        color="red",
                    )

    _plt.tight_layout()

    plots_dir = os.path.join(self.storage_path, "plots", save_subdir)
    os.makedirs(plots_dir, exist_ok=True)
    fig.savefig(
        os.path.join(plots_dir, f"phase_{self._current_phase_id:03d}.png"),
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        os.path.join(plots_dir, f"phase_{self._current_phase_id:03d}.svg"),
        bbox_inches="tight",
    )
    _plt.show()


agent._plot_live = types.MethodType(_plot_live_4d, agent)
print(
    f"Live-plot override installed: 3-row x "
    f"{len(np.unique(agent.x_total_linespace[:, 2]))}-pulse_interval panel grid "
    f"(led_power pinned at global-best per phase)."
)

In [ ]:
# --- Run the composed (FOV finder + BO) loop ---
# Each phase: FOVFinder picks 18 fresh FOVs (3 FOVs in 6 fresh wells), then
# OscillationBO runs one batch BO iteration over those FOVs.  The GP
# accumulates observations across all phases.


# Disconnect napari live view so it does not interfere with the acquisition engine
try:
    mm_wdg._core_link.cleanup()
except:
    pass

sleep_time = 3600 * 10  # sleep until tomorrow morning
for t in range(0, sleep_time):
    time.sleep(1)


composed_agent.run()

In [ ]:
# Post-processing: merge per-FOV tracks
utils.generate_exp_data_from_tracks(path)

## Visualise results

In [ ]:
if agent.df_results is not None and not agent.df_results.empty:
    df = agent.df_results
    obj_name = agent.objective_metric.name

    # Per-pulse_interval panel grid:
    #   row 0 -- per-FOV scatter (stim_exposure x ramp colored by objective)
    #   row 1 -- per-condition mean across FOVs (size = n_fovs in that condition)
    pi_levels = sorted(df["pulse_interval"].astype(int).unique().tolist())
    n_levels = len(pi_levels)

    vmin = float(df[obj_name].min())
    vmax = float(df[obj_name].max())

    fig, axes = plt.subplots(
        2, n_levels, figsize=(5 * n_levels, 10), squeeze=False, sharex=True, sharey=True
    )

    for col, pi in enumerate(pi_levels):
        df_pi = df[df["pulse_interval"].astype(int) == pi]

        ax = axes[0, col]
        if not df_pi.empty:
            sc = ax.scatter(
                df_pi["stim_exposure"],
                df_pi["ramp"],
                c=df_pi[obj_name],
                cmap="viridis",
                s=40,
                edgecolors="k",
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
            )
            fig.colorbar(sc, ax=ax, label=obj_name)
        ax.set_xlabel("stim_exposure (ms)")
        if col == 0:
            ax.set_ylabel("ramp (ms/pulse)")
        ax.set_title(f"Per-FOV  |  pulse_interval={pi}\n(n_FOVs={len(df_pi)})")

        ax = axes[1, col]
        cond_agg = (
            df_pi.groupby(["stim_exposure", "ramp", "led_power"])
            .agg(mean_obj=(obj_name, "mean"), n_fovs=(obj_name, "count"))
            .reset_index()
        )
        if not cond_agg.empty:
            sc = ax.scatter(
                cond_agg["stim_exposure"],
                cond_agg["ramp"],
                c=cond_agg["mean_obj"],
                s=cond_agg["n_fovs"] * 30,
                cmap="viridis",
                edgecolors="k",
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
            )
            fig.colorbar(sc, ax=ax, label=f"mean {obj_name}")
        ax.set_xlabel("stim_exposure (ms)")
        if col == 0:
            ax.set_ylabel("ramp (ms/pulse)")
        ax.set_title(
            f"Per-condition mean  |  pulse_interval={pi}\n(marker size = n_FOVs)"
        )

    fig.suptitle(
        f"Measured {obj_name} per pulse_interval level\n"
        "(BO axes: stim_exposure x ramp; led_power not panelled)",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    # --- Convergence ---
    if agent._iteration_means:
        fig, ax = plt.subplots(figsize=(8, 5))
        means = np.array(agent._iteration_means)
        cumulative_best = np.maximum.accumulate(means)
        ax.plot(means, "o-", alpha=0.5, label="iteration mean")
        ax.plot(cumulative_best, "s-", color="red", label="cumulative best")
        ax.set_xlabel("Iteration")
        ax.set_ylabel(obj_name)
        ax.set_title("BO convergence")
        ax.legend()
        plt.tight_layout()
        plt.show()

    # --- led_power 1D summary ---
    # Marginal effect of LED power on the objective: groupby led_power,
    # mean across all (stim_exposure, ramp, pulse_interval) trials.  Shows
    # the LED-saturation curve the BO is exploiting.
    fig, ax = plt.subplots(figsize=(8, 5))
    pwr_agg = (
        df.groupby("led_power")
        .agg(
            mean_obj=(obj_name, "mean"),
            std_obj=(obj_name, "std"),
            n_fovs=(obj_name, "count"),
        )
        .reset_index()
    )
    ax.errorbar(
        pwr_agg["led_power"],
        pwr_agg["mean_obj"],
        yerr=pwr_agg["std_obj"].fillna(0.0),
        fmt="o-",
        capsize=4,
    )
    ax.set_xscale("log")  # log axis matches log_scale=True on the BO param
    ax.set_xlabel("led_power (%, log scale)")
    ax.set_ylabel(f"mean {obj_name}")
    ax.set_title(
        f"LED power vs {obj_name} (marginal across all other axes)\n"
        "(error bars = std across FOVs)"
    )
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    # --- Best condition (full 4-axis grid) ---
    cond_agg_all = (
        df.groupby(["stim_exposure", "ramp", "pulse_interval", "led_power"])
        .agg(mean_obj=(obj_name, "mean"), n_fovs=(obj_name, "count"))
        .reset_index()
    )
    best_idx = cond_agg_all["mean_obj"].idxmax()
    best = cond_agg_all.loc[best_idx]
    print(f"\nBest condition by {obj_name} (mean across FOVs):")
    print(f"  stim_exposure  = {best['stim_exposure']:.0f} ms")
    print(f"  ramp           = {best['ramp']:.0f} ms/pulse")
    print(f"  pulse_interval = {int(best['pulse_interval'])} frames")
    print(f"  led_power      = {int(best['led_power'])} %")
    print(f"  mean {obj_name:<13}= {best['mean_obj']:.4f}")
    print(f"  tested on {int(best['n_fovs'])} FOVs")
else:
    print("No data collected yet.")

In [ ]:
# GP-predicted landscape for v10, marginalised over covariates AND
# led_power (since rendering 5 pulse_interval x 10 led_power = 50 panels
# is overwhelming).  We average GP predictions over both joint covariate
# samples and the empirical led_power distribution observed so far.
import gpax
import gpax.utils
from faro.agents.bo_optimization_sparse import _safe_batch_size

if agent.model is not None and agent.x is not None:
    obj_name = agent.objective_metric.name
    rng_key, rng_key_pred = gpax.utils.get_keys()

    exposure_vals = np.arange(25.0, 1000.0 + 25.0, 25.0)  # 40 levels
    ramp_vals = np.arange(0.0, 50.0 + 10.0, 10.0)  # 6 levels
    pi_levels = sorted(np.unique(agent.x_total_linespace[:, 2]).astype(int).tolist())
    pwr_levels = sorted(np.unique(agent.x_total_linespace[:, 3]).astype(int).tolist())
    n_pi = len(pi_levels)
    n_e = len(exposure_vals)
    n_r = len(ramp_vals)
    n_p = len(pwr_levels)

    df = agent.df_results
    n_cov_samples = 50
    cov_vals_full = df[[c.name for c in agent.bo_covariates]].to_numpy(dtype=float)
    if cov_vals_full.shape[0] == 0:
        raise RuntimeError("No covariate observations available for marginalisation.")
    _rng = np.random.default_rng(0)
    idx = _rng.integers(0, cov_vals_full.shape[0], size=n_cov_samples)
    cov_samples = cov_vals_full[idx]

    # Combined ctrl_grid spanning all (pulse_interval, led_power) combos.
    # Layout: outer pi, then power, then exposure, then ramp.
    ctrl_grid = np.array(
        [
            [e, r, float(pi), float(pwr)]
            for pi in pi_levels
            for pwr in pwr_levels
            for e in exposure_vals
            for r in ramp_vals
        ]
    )
    n_grid = ctrl_grid.shape[0]

    x_full = np.hstack(
        [
            np.repeat(ctrl_grid, n_cov_samples, axis=0),
            np.tile(cov_samples, (n_grid, 1)),
        ]
    )
    x_full_scaled = agent._x_scaler.transform(x_full)

    n_rows = np.asarray(x_full_scaled).shape[0]
    bs = _safe_batch_size(n_rows, 2000)
    y_pred_scaled, _ = agent.model.predict_in_batches(
        rng_key_pred,
        x_full_scaled,
        batch_size=bs,
        n=1,
        noiseless=True,
    )
    y_pred = agent._y_scaler.inverse_transform(
        np.asarray(y_pred_scaled).reshape(-1, 1)
    ).flatten()

    Y_marg = y_pred.reshape(n_grid, n_cov_samples).mean(axis=1)
    # axes after reshape: (pi, pwr, exposure, ramp)
    Y_4d = Y_marg.reshape(n_pi, n_p, n_e, n_r)

    # Marginalise over led_power (mean) for the per-pulse_interval surfaces.
    Y_3d = Y_4d.mean(axis=1)  # (pi, exposure, ramp)
    pi_optima = {}
    for i, pi in enumerate(pi_levels):
        flat_idx = int(np.argmax(Y_3d[i]))
        e_idx, r_idx = np.unravel_index(flat_idx, (n_e, n_r))
        pi_optima[pi] = (
            float(exposure_vals[e_idx]),
            float(ramp_vals[r_idx]),
            float(Y_3d[i, e_idx, r_idx]),
        )

    vmin = float(Y_3d.min())
    vmax = float(Y_3d.max())
    global_best_pi = max(pi_levels, key=lambda p: pi_optima[p][2])

    fig, axes = plt.subplots(
        1, len(pi_levels), figsize=(7 * len(pi_levels), 6), squeeze=False, sharey=True
    )
    for col, pi in enumerate(pi_levels):
        ax = axes[0, col]
        Y_grid = Y_3d[col]
        cf = ax.contourf(
            ramp_vals,
            exposure_vals,
            Y_grid,
            levels=20,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )
        fig.colorbar(cf, ax=ax, label=f"predicted {obj_name}")

        df_pi = df[df["pulse_interval"].astype(int) == pi]
        if not df_pi.empty:
            ax.scatter(
                df_pi["ramp"],
                df_pi["stim_exposure"],
                c=df_pi[obj_name],
                cmap="viridis",
                s=30,
                edgecolors="white",
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
                zorder=5,
            )

        opt_e, opt_r, opt_val = pi_optima[pi]
        is_global = pi == global_best_pi
        ax.scatter(
            opt_r,
            opt_e,
            c="red",
            s=240 if is_global else 180,
            marker="*",
            edgecolors="black",
            linewidths=2,
            zorder=10,
            label=(
                f"opt: exp={opt_e:.0f}ms, ramp={opt_r:.0f}\n"
                f"pred={opt_val:.3f}{'  (global)' if is_global else ''}"
            ),
        )
        ax.set_xlabel("ramp (ms/pulse)")
        if col == 0:
            ax.set_ylabel("stim_exposure (ms)")
        ax.set_title(f"pulse_interval = {pi}")
        ax.legend(loc="lower right", fontsize=7)

    fig.suptitle(
        f"GP-predicted {obj_name} per pulse_interval\n"
        f"(marginalised over led_power AND {n_cov_samples} joint covariate samples)",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    # --- Marginal effect of led_power on the GP surface ---
    # Average over (pulse_interval, exposure, ramp) for each led_power,
    # to visualise the LED saturation curve the BO is fitting.
    Y_pwr = Y_4d.mean(axis=(0, 2, 3))  # axis-aligned mean -> (n_p,)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(pwr_levels, Y_pwr, "o-", lw=1.5)
    ax.set_xscale("log")
    ax.set_xlabel("led_power (%, log scale)")
    ax.set_ylabel(f"GP-predicted mean {obj_name}")
    ax.set_title("LED power saturation curve (GP marginal)")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    print(
        f"GP-predicted optimum per pulse_interval (led_power and covariates marginalised):"
    )
    for pi in pi_levels:
        opt_e, opt_r, opt_val = pi_optima[pi]
        marker = "  <- global" if pi == global_best_pi else ""
        print(
            f"  pulse_interval={pi}: stim_exposure={opt_e:>5.0f}ms  "
            f"ramp={opt_r:>4.0f}ms  predicted={obj_name}={opt_val:.4f}{marker}"
        )
    bp = pi_optima[global_best_pi]
    print(
        f"\nGlobal optimum: pulse_interval={global_best_pi}, "
        f"stim_exposure={bp[0]:.0f}ms, ramp={bp[1]:.0f}ms, "
        f"predicted {obj_name}={bp[2]:.4f}"
    )

In [ ]:
# Covariate effects on the configured BO objective (per FOV)
if agent.df_results is not None and not agent.df_results.empty:
    df = agent.df_results
    obj_name = agent.objective_metric.name

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].scatter(df["n_cells"], df[obj_name], alpha=0.6, s=20)
    axes[0].set_xlabel("n_cells per FOV")
    axes[0].set_ylabel(obj_name)
    axes[0].set_title(f"Cell density vs {obj_name}")

    axes[1].scatter(df["optortk_expression"], df[obj_name], alpha=0.6, s=20)
    axes[1].set_xlabel("optoRTK expression (mCitrine intensity)")
    axes[1].set_ylabel(obj_name)
    axes[1].set_title(f"optoRTK expression vs {obj_name}")

    axes[2].scatter(df["baseline_cnr"], df[obj_name], alpha=0.6, s=20)
    axes[2].set_xlabel("baseline_cnr (pre-stim CNR)")
    axes[2].set_ylabel(obj_name)
    axes[2].set_title(f"Baseline CNR vs {obj_name}")

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import glob, os

# Per-phase delta: each checkpoint stores cumulative df_results, so
# phase N's new conditions = rows in ckpt N that aren't in ckpt N-1.
RUN_PATH = path
ckpts = sorted(
    glob.glob(os.path.join(RUN_PATH, "checkpoints", "bo_results_phase_*.parquet"))
)

prev_len = 0
summary = []
for f in ckpts:
    phase_id = int(os.path.basename(f).split("_")[-1].split(".")[0])
    df = pd.read_parquet(f)
    df_new = df.iloc[prev_len:]
    prev_len = len(df)
    conds = (
        df_new[["stim_exposure", "ramp", "pulse_interval", "led_power"]]
        .drop_duplicates()
        .values.tolist()
    )
    summary.append(
        {
            "phase_id": phase_id,
            "n_fovs": len(df_new),
            "conditions": [
                f"(exp={int(s)}ms, ramp={int(r)}, pi={int(p)}, pwr={int(pwr)}%)"
                for s, r, p, pwr in conds
            ],
        }
    )

for row in summary:
    print(
        f"Phase {row['phase_id']:2d}  ({row['n_fovs']:2d} FOVs): "
        f"{'  |  '.join(row['conditions'])}"
    )